# TP4 — De píxeles a metros (informe)

**Nombre:** Matías · **Entrega:** _fecha_

Usa `mv.camara` (este TP) y `mv.geometria` (TP1: `aplicar`, `warp`, `recta_por_dos_puntos`, `interseccion`).
OpenCV aparece **solo para comparar**. Respondé en las celdas **✍️ Respuesta**.

Las celdas sin **TODO** ya están hechas (cámara sintética, dibujo de la cancha, medición del error en metros): leelas, porque las vas a usar.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from mv import camara as cam
from mv import geometria as g
from mv import datos

ANCHO_IMG, ALTO_IMG = 1920, 1080

In [ ]:
def camara_club(C=(50.0, 80.0, -9.0), objetivo=(50.0, 34.0, 0.0), f=1400.0):
    """K, R, t de una cámara en C (Z hacia ABAJO: −9 = 9 m de altura) que mira a `objetivo`, sin rolido."""
    K = np.array([[f, 0, ANCHO_IMG / 2], [0, f, ALTO_IMG / 2], [0, 0, 1.0]])
    C = np.asarray(C, float)
    zc = np.asarray(objetivo, float) - C; zc /= np.linalg.norm(zc)        # adelante
    abajo = np.array([0, 0, 1.0])
    yc = abajo - (abajo @ zc) * zc; yc /= np.linalg.norm(yc)              # abajo en la imagen
    xc = np.cross(yc, zc)                                                 # derecha en la imagen
    R = np.vstack([xc, yc, zc])
    return K, R, -R @ C


def lineas_cancha(largo=100.0, ancho=68.0, n=60):
    """Las líneas de la cancha como lista de polilíneas (k, 2) en metros (mismas convenciones que modelo_cancha)."""
    L, A = largo, ancho
    rect = lambda x0, y0, x1, y1: np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1], [x0, y0]], float)
    t = np.linspace(0, 2 * np.pi, n)
    lineas = [rect(0, 0, L, A), np.array([[L / 2, 0], [L / 2, A]]),
              np.column_stack([L / 2 + 9.15 * np.cos(t), A / 2 + 9.15 * np.sin(t)])]
    for x0, s in [(0, 1), (L, -1)]:
        lineas.append(rect(x0, A / 2 - 20.16, x0 + s * 16.5, A / 2 + 20.16))   # área grande
        lineas.append(rect(x0, A / 2 - 9.16, x0 + s * 5.5, A / 2 + 9.16))      # área chica
    return lineas


def dibujar_cancha(ax, H, color="yellow", lw=1.2, **kw):
    """Dibuja la cancha llevada a la imagen con H (cancha → imagen). Interpola cada segmento en metros,
    porque una recta en la imagen sigue siendo recta, pero el círculo no sigue siendo círculo."""
    for pl in lineas_cancha():
        dens = np.concatenate([np.linspace(a, b, 30) for a, b in zip(pl[:-1], pl[1:])])
        uv = g.aplicar(H, dens)
        ax.plot(uv[:, 0], uv[:, 1], color=color, lw=lw, **kw)


def visibles(H, puntos, margen=0):
    """Máscara de los puntos (en metros) que caen dentro de la imagen de 1920×1080."""
    uv = g.aplicar(H, puntos)
    return ((uv[:, 0] >= margen) & (uv[:, 0] < ANCHO_IMG - margen) &
            (uv[:, 1] >= margen) & (uv[:, 1] < ALTO_IMG - margen))


def error_en_metros(H_est, H_real, puntos_m):
    """Error (m) de H_est sobre puntos de la cancha: el píxel VERDADERO de cada punto se lleva de vuelta a la
    cancha con H_est⁻¹ y se mide cuánto se aleja del punto original. Es lo que importa: ¿cuántos metros me equivoco?"""
    uv = g.aplicar(H_real, puntos_m)
    return np.linalg.norm(g.aplicar(np.linalg.inv(H_est), uv) - puntos_m, axis=1)


gx, gy = np.meshgrid(np.linspace(0, 100, 41), np.linspace(0, 68, 29))
GRILLA = np.column_stack([gx.ravel(), gy.ravel()])     # puntos de la cancha donde se mide el error

PC = cam.modelo_cancha()
NOMBRES = list(PC)
PUNTOS = np.array([PC[k] for k in NOMBRES])

---
## 1. La cámara sintética

Una cámara parecida a la del club: al costado de la cancha, a 9 m de altura y 12 m afuera de la lateral, mirando el centro.

In [ ]:
K, R, t = camara_club()
P = cam.matriz_camara(K, R, t)
H = cam.homografia_desde_camara(P)

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.add_patch(plt.Rectangle((0, 0), ANCHO_IMG, ALTO_IMG, fc="#3a7d3a"))
dibujar_cancha(ax, H, color="white")
vis = visibles(H, PUNTOS)
uv = g.aplicar(H, PUNTOS[vis])
ax.plot(uv[:, 0], uv[:, 1], "r.", ms=6)
ax.set_xlim(-200, ANCHO_IMG + 200); ax.set_ylim(ALTO_IMG + 100, -100); ax.set_aspect("equal")
ax.set_title(f"La cancha vista por la cámara sintética ({vis.sum()} de {len(PUNTOS)} puntos notables dentro del cuadro)")
plt.show()
print(np.round(H, 4))

In [ ]:
# TODO 1a: el punto de fuga de las líneas en dirección Y (las de fondo, los costados de las áreas).
#   - Llevá a la imagen los extremos de dos de esas líneas (por ejemplo, los costados "frente" de las dos áreas grandes)
#     y cortá las rectas con g.recta_por_dos_puntos y g.interseccion.
#   - Compará con la fórmula del apunte 01: v ≃ K R d, con d = (0, 1, 0).
#   - ¿Y el de las laterales (d = (1, 0, 0))? ¿Por qué da (casi) en el infinito?

# TODO 1b: altura en píxeles de un jugador de 1,75 m parado en (50, y), para y en np.linspace(0, 68, 35).
#   Proyectá los pies (Z = 0) y la cabeza (Z = −1,75: Z va hacia abajo) con cam.proyectar y graficá altura vs y.

**Preguntas**
1. ¿Dónde está el punto de fuga de las líneas en dirección Y? ¿Coinciden los dos cálculos?
2. ¿Entre qué valores está la altura del jugador en píxeles? Compará con los **30–230 px** medidos en los videos del club. ¿Qué te dice sobre esta cámara sintética comparada con la real (focal, distancia)?

✍️ **Respuesta:**

---
## 2. DLT, OpenCV y normalización

In [ ]:
import cv2

GRILLA_VIS = GRILLA[visibles(H, GRILLA)]                  # solo la parte de la cancha que esta cámara ve

r = np.random.default_rng(0)
src = GRILLA_VIS[r.choice(len(GRILLA_VIS), 15, replace=False)]
dst = g.aplicar(H, src) + r.normal(0, 1.5, (15, 2))     # clicks con ~1,5 px de error

H_mio = cam.dlt_homografia(src, dst)
H_cv, _ = cv2.findHomography(src, dst, 0)
for nombre, Hx in [("mío (DLT)", H_mio), ("OpenCV", H_cv)]:
    print(f"{nombre:10s}  reproyección media = {cam.error_reproyeccion(Hx, src, dst).mean():.2f} px"
          f" | error en la cancha: mediana {np.median(error_en_metros(Hx, H, GRILLA_VIS)):.3f} m")

In [ ]:
# TODO 2: el efecto de normalizar. Siempre: 10 puntos de GRILLA_VIS al azar, error en metros sobre GRILLA_VIS.
#   (a) σ₁/σ₈ de matriz_dlt con y sin normalizar (mediana de 300 sorteos).
#   (b) 300 ensayos con 2 px de ruido: error mediano en metros con normalizar=True y con normalizar=False.
#   (c) Lo mismo SIN ruido (correspondencias exactas): ¿cuánto error queda, y de dónde sale?
#   (d) Repetí (b) y (c) haciendo la SVD en float32: escribí ACÁ una dlt de prueba que haga
#       A = cam.matriz_dlt(...).astype(np.float32) antes de np.linalg.svd (no toques mv/camara.py).
#   Armá una tabla: filas (b), (c), (d); columnas con y sin normalizar.

**Preguntas**
3. ¿Cuánto mejora el condicionamiento? ¿Cuánto cambia el error **numérico** (sin ruido)? ¿Y el error final **con** ruido? ¿Y en float32?
4. Con lo que mediste: ¿qué *hace* la normalización, y qué **no**? ¿En qué situación esperarías que sí cambie el resultado final? (Apunte 03, sección 3.)
5. ¿Por qué OpenCV queda un poco mejor que tu DLT en error de reproyección?

✍️ **Respuesta:**

---
## 3. Estabilidad: cuántos puntos y dónde

La pregunta del proyecto anterior (D24): ¿se puede calibrar con 4 puntos? Ahora se mide.

La cámara del club **panea** siguiendo la pelota. Para esta sección, la misma cámara sintética gira hacia el **área izquierda**
(como cuando la jugada está ahí, que es cuando hay más marcas a la vista). Los "clicks" son los puntos notables **visibles**
de `modelo_cancha`, con ruido gaussiano de 2 px. El error se mide en metros sobre la parte **visible** de la cancha.

In [ ]:
K_a, R_a, t_a = camara_club(objetivo=(20.0, 34.0, 0.0))
H_area = cam.homografia_desde_camara(cam.matriz_camara(K_a, R_a, t_a))
GRILLA_AREA = GRILLA[visibles(H_area, GRILLA)]
idx_vis = np.flatnonzero(visibles(H_area, PUNTOS, margen=20))
print(f"{len(idx_vis)} puntos notables visibles:", [NOMBRES[i] for i in idx_vis])

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.add_patch(plt.Rectangle((0, 0), ANCHO_IMG, ALTO_IMG, fc="#3a7d3a"))
dibujar_cancha(ax, H_area, color="white")
uv = g.aplicar(H_area, PUNTOS[idx_vis])
ax.plot(uv[:, 0], uv[:, 1], "r.", ms=7)
ax.set_xlim(0, ANCHO_IMG); ax.set_ylim(ALTO_IMG, 0); ax.set_aspect("equal")
ax.set_title("La cámara paneada hacia el área izquierda")
plt.show()


def un_ensayo(idx, ruido_px, rng, grilla=GRILLA_AREA):
    """Calibra con los puntos idx (+ ruido en los clicks) y devuelve el error mediano en metros sobre `grilla`.
    Si la configuración es degenerada (la H sale con infinitos o no se puede invertir), devuelve inf."""
    src = PUNTOS[idx]
    dst = g.aplicar(H_area, src) + rng.normal(0, ruido_px, (len(idx), 2))
    try:
        He = cam.dlt_homografia(src, dst)
        if not np.all(np.isfinite(He)):
            return np.inf
        return np.median(error_en_metros(He, H_area, grilla))
    except np.linalg.LinAlgError:
        return np.inf


r = np.random.default_rng(1)
print("ejemplo, 6 puntos al azar:", un_ensayo(r.choice(idx_vis, 6, replace=False), 2.0, r))

In [ ]:
# TODO 3a: para n en [4, 5, 6, 8, 12] y 300 ensayos cada uno (n puntos de idx_vis al azar, 2 px de ruido):
#   mediana y percentil 90 del error, y % de ensayos con más de 5 m. Tabla + boxplot (escala log en y).
#   Mirá los peores ensayos con 5 puntos: ¿qué tienen en común? (Pista: contá cuántos puntos hay sobre la línea de fondo.)

# TODO 3b: "amontonados" contra "repartidos", con 6 puntos:
#   - amontonados: solo los del área izquierda (nombres que empiezan con "area" o "penal");
#   - repartidos: cualquiera de idx_vis.
#   Para cada caso, error sobre GRILLA_AREA y sobre la parte con X < 20 (GRILLA_AREA[GRILLA_AREA[:, 0] < 20]).

# TODO 3c: un mapa del error sobre la cancha (scatter de GRILLA_AREA coloreado por error_en_metros)
#   para UN ensayo con 4 puntos y otro con 8. ¿Dónde se equivoca más?

**Preguntas**
6. ¿Cuántos puntos hacen falta para que el error mediano baje de 1 m? ¿Y el percentil 90?
7. ¿Por qué con 4 puntos el percentil 90 es tan malo, si el error de reproyección en esos 4 puntos es 0? ¿Qué tenían los peores ensayos de 5 puntos?
8. Con los puntos amontonados en un área, ¿qué tan bien medirías la ubicación de un **tiro** (que está en esa área)? ¿Y la posición de un mediocampista cerca de la línea de medio?

✍️ **Respuesta:**

---
## 4. RANSAC

In [ ]:
r = np.random.default_rng(2)
src = GRILLA_VIS[r.choice(len(GRILLA_VIS), 30, replace=False)]
dst = g.aplicar(H, src) + r.normal(0, 1.0, (30, 2))
dst[:3] += r.uniform(80, 250, (3, 2))                    # 3 clicks malos de 30

H_dlt = cam.dlt_homografia(src, dst)
H_rs, inl = cam.ransac_homografia(src, dst, umbral=4.0, rng=np.random.default_rng(0))
print(f"DLT con todo: error mediano {np.median(error_en_metros(H_dlt, H, GRILLA_VIS)):.2f} m")
print(f"RANSAC:       error mediano {np.median(error_en_metros(H_rs, H, GRILLA_VIS)):.2f} m | outliers detectados: {np.flatnonzero(~inl)}")

In [ ]:
# TODO 4: ¿la fórmula de iteraciones predice bien?
#   Con 40 correspondencias y w = 0,5 (20 outliers): para N en [5, 10, 20, 40, 72, 120],
#   simulá 2000 veces "elegir N muestras de 4 índices al azar (sin repetir dentro de la muestra)" y medí la fracción de
#   veces en que NINGUNA muestra es toda de inliers. Graficá contra la teoría (1 − w⁴)^N.
#   ¿Por qué la simulación falla un poco MÁS que la teoría? (Guía, E3.)

**Preguntas**
9. ¿Cuánto empeoran 3 outliers el DLT común? ¿RANSAC encontró exactamente los 3?
10. ¿La fórmula predice bien la tasa de fallas? ¿Por qué la diferencia va en esa dirección?

✍️ **Respuesta:**

---
## 5. Un frame real de Hebraica

Cargá tus clicks (instrucciones en el enunciado). Los nombres tienen que ser los de `modelo_cancha`.

In [ ]:
frame = datos.cargar_frame()            # el primero de datos/frames/ (o uno sintético si no hay)
fig, ax = plt.subplots(figsize=(12, 6.75))
ax.imshow(frame)
ax.set_xticks(np.arange(0, frame.shape[1], 100)); ax.set_yticks(np.arange(0, frame.shape[0], 100))
ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
ax.set_title("Grilla cada 100 px para ubicarte")
plt.show()
print("nombres válidos:", NOMBRES)

In [ ]:
clicks = {
    # "area_grande_izq_lejana_frente": (812, 274),     ← ejemplo de formato: nombre → (u, v) en píxeles
}

H_real = None
if len(clicks) >= 4:
    nombres_usados = list(clicks)
    src_real = np.array([PC[k] for k in nombres_usados])
    dst_real = np.array([clicks[k] for k in nombres_usados], float)
    H_real = cam.dlt_homografia(src_real, dst_real)
    res = cam.error_reproyeccion(H_real, src_real, dst_real)
    for k, e in zip(nombres_usados, res):
        print(f"  {k:32s} residuo {e:6.2f} px")

    fig, ax = plt.subplots(figsize=(12, 6.75))
    ax.imshow(frame)
    dibujar_cancha(ax, H_real)
    ax.plot(dst_real[:, 0], dst_real[:, 1], "r+", ms=12, mew=2)
    ax.set_xlim(0, frame.shape[1]); ax.set_ylim(frame.shape[0], 0)
    ax.set_title(f"Cancha reproyectada con {len(clicks)} puntos")
    plt.show()
else:
    print(f"Faltan clicks: hay {len(clicks)} y hacen falta al menos 4 (y 6 o más para poder validar).")

In [ ]:
# TODO 5: validación dejando uno afuera (leave-one-out), EN METROS.
#   Para cada punto i: ajustá H con todos los demás, llevá el click i a la cancha con H⁻¹ y medí la distancia al
#   punto verdadero PC[nombre_i]. Tabla con el error de cada punto. (Con 4 puntos no se puede: ¿por qué?)
#   Además: ¿cuántos metros es 1 píxel en la lateral lejana y en la cercana? (Mové 1 px un punto de cada zona y
#   llevalo a la cancha.)

**Preguntas**
11. ¿Qué puntos usaste y cuáles te dejaron dudas? ¿Coincide el residuo grande con el punto dudoso?
12. ¿Cuánto da el error *leave-one-out* en metros? ¿Dónde es peor? Comparalo con lo que predijo la sección 3 para esa cantidad de puntos.
13. ¿Las líneas reproyectadas caen sobre la cal? ¿En qué zona se separan?

✍️ **Respuesta:**

---
## 6. Vista cenital y posiciones de jugadores

In [ ]:
PX_POR_M, MARGEN_M = 10, 4
S = np.array([[PX_POR_M, 0, MARGEN_M * PX_POR_M],        # metros de la cancha → píxeles de la vista cenital
              [0, PX_POR_M, MARGEN_M * PX_POR_M],
              [0, 0, 1.0]])
forma_cenital = ((68 + 2 * MARGEN_M) * PX_POR_M, (100 + 2 * MARGEN_M) * PX_POR_M)

if H_real is not None:
    imagen_a_cenital = S @ np.linalg.inv(H_real)          # imagen → cancha (m) → vista cenital (px)
    cenital = g.warp(frame, imagen_a_cenital, forma_cenital)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(cenital)
    dibujar_cancha(ax, S, color="yellow", lw=0.8)        # S lleva metros a la vista cenital
    ax.set_title("Vista cenital (tu warp del TP1)")
    plt.show()
else:
    print("Primero la sección 5.")

In [ ]:
# TODO 6: clickeá los PIES (borde inferior de la caja) de 5–10 jugadores en el frame → `pies` (k, 2) en píxeles.
#   Llevalos a metros con g.aplicar(np.linalg.inv(H_real), pies) y dibujalos sobre la vista cenital
#   (acordate de pasar por S). Hacé lo mismo con la CABEZA de 2 de ellos: ¿cuántos metros se corre? Compará con la
#   cuenta de la guía (B4).

**Preguntas**
14. ¿La vista cenital se ve rectangular? ¿Qué se ve deformado y por qué (pensá en qué **no** está en el plano Z = 0)?
15. ¿Las posiciones de los jugadores son creíbles? ¿Cuánto se corre un jugador si clickeás la cabeza en vez de los pies?

✍️ **Respuesta:**

---
## 7. Conclusión

16. Con la calibración de la sección 5: un jugador se mueve durante 90 minutos y su posición se mide 10 veces por segundo. Si cada posición tiene un error del orden del *leave-one-out* que mediste, ¿cuánto error tendría su distancia recorrida? (Guía, D3. Ojo: **no** se cancela.)
17. ¿Qué cambiaría si la cámara fuera **fija**? ¿Y si panea, como la del club?

En 5–10 líneas: qué te llevás de U4 y qué te quedó flojo.

✍️ **Respuesta:**